# What Predicts Earnings? A Linear Regression Analysis Using PSID 2017 Data
### 📝 SKELETON NOTEBOOK — fill in the `# TODO` sections

**Business question:** Among working-age adults who report positive labor income, how do
education, work experience (proxied by age), gender, marital status, and educational
attainment relate to annual labor income? Can you build a linear regression model that
explains and predicts income from these characteristics?

**Dataset:** `psid_2017.csv` — a household-roster extract from the 2017 wave of the
Panel Study of Income Dynamics (PSID).

| Column | Description |
|---|---|
| `X` | Row / person identifier |
| `gender` | 1 = Male, 2 = Female |
| `age` | Age in years (999 = missing/NA code used by PSID) |
| `married` | Marital status code (0 = never married, 1 = married, 2 = widowed/separated, 3 = divorced) |
| `employed` | Employment status text, or missing for people too young / not asked |
| `educated_in_us` | Code for country/context of education (1 = US; other codes = non-US / don't know / inapplicable) |
| `highest_degree` | Highest degree earned (text categories) |
| `education_years` | Years of completed education (99 = missing/refused code) |
| `labor_income` | Annual labor income in US dollars |

This project follows the four-stage statistical workflow: **(1) confirm data
assumptions, (2) build a model on training data, (3) assess model fit, (4) analyze model
results** — the same workflow from *Linear Regression in R*, applied here in Python to a
real, messy government survey extract.

> 💡 A completed reference solution is available in `02_project_solution.ipynb` if you
> get stuck — but try each step yourself first! A `03_project_cheatsheet.ipynb` is also
> available for quick syntax lookups.

## Step 0 — Setup
Import the libraries you'll need. At minimum: `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `statsmodels`, `scipy.stats`, and `sklearn.model_selection.train_test_split`.

In [ ]:
# TODO: import pandas, numpy, matplotlib.pyplot, seaborn, statsmodels.api,
# statsmodels.formula.api, scipy.stats, and train_test_split / metrics from sklearn.
# Also set a random seed for reproducibility.


## Step 1 — Load and Inspect the Data

**Instructions:**
1. Load `psid_2017.csv` into a DataFrame called `raw`. (Hint: the first, unnamed column
   is a row index — use `index_col=0`.)
2. Use `.shape`, `.info()`, `.head()`, and `.describe(include="all")` to get a full
   picture of the data: how many rows/columns, what types, what ranges.
3. Write down (in a markdown cell) at least two things that look suspicious or need
   further investigation before modeling.

In [ ]:
# TODO: load the CSV into `raw` with the correct index_col


In [ ]:
# TODO: inspect shape / info


In [ ]:
# TODO: inspect head() and describe(include='all')


**Your observations:** *(replace this text)*
- ...
- ...

## Step 2 — Data Cleaning

### 2.1 Replace sentinel/missing codes with `NaN`

PSID encodes missing data using out-of-range numeric codes, not blanks.

**Instructions:**
1. Copy `raw` into a new DataFrame called `df`.
2. Count how many rows have `age == 999`. Replace those values with `np.nan`.
3. Count how many rows have `education_years == 99`. Replace those values with `np.nan`.

In [ ]:
# TODO: df = raw.copy()
# TODO: count and replace age == 999 with NaN
# TODO: count and replace education_years == 99 with NaN


### 2.2 Restrict to the working-age population

**Instructions:**
1. Filter `df` down to rows where `age` is between 25 and 65 (inclusive). Save the
   result as `working_age`.
2. Print the number of rows before and after this filter.
3. Look at the `value_counts()` of `employed` within `working_age` — does it look more
   sensible now than it did for the full dataset?

In [ ]:
# TODO: create `working_age` by filtering df on age


### 2.3 Investigate `labor_income`

**Instructions:**
1. Within `working_age`, calculate what fraction of rows have `labor_income == 0`.
2. Plot a histogram of `labor_income` for `working_age`. What does the shape tell you?
3. **Think about it:** is a $0 income value here likely to be *real* (the person truly
   earned nothing) or a sign of something structural about how this dataset was built?
   (Hint: look at what fraction of people whose `employed` status is literally
   `"employed"` still show `labor_income == 0` — is that plausible?)
4. Based on your answer, decide how to filter the data down to a sensible **analytic
   sample** for modeling income, and save it as `analysis_df`. Also drop any rows still
   missing `education_years` at this point.

In [ ]:
# TODO: compute the fraction of working_age rows with labor_income == 0


In [ ]:
# TODO: histogram of labor_income for working_age


**Your reasoning and decision:** *(replace this text — explain what you decided to filter on and why)*

In [ ]:
# TODO: build `analysis_df` — your cleaned, filtered analytic sample
# TODO: print the final sample size


### 2.4 Recode categorical variables

**Instructions:**
1. Create a `gender_label` column mapping 1 → `"Male"`, 2 → `"Female"`.
2. Create a `married_label` column: `"Married"` if `married == 1`, else `"Not married"`.
3. Create a `degree_tier` column that collapses `highest_degree` into three ordered
   groups: `"No college"`, `"Some college / Associate"`, `"Graduate / Professional"`.
   Decide what to do with the `"refused_answer"` category (hint: consider dropping it).
4. Create an `educated_in_us_label` column: `"Yes"` if `educated_in_us == 1`, else
   `"No/Unknown"`.

In [ ]:
# TODO: build gender_label, married_label, degree_tier, educated_in_us_label


## Step 3 — Checking the Linearity Assumption

**Instructions:**
1. Make a scatterplot of `education_years` (x) vs `labor_income` (y) using
   `analysis_df`. Describe the shape.
2. Compute the Pearson correlation coefficient between the two variables
   (`scipy.stats.pearsonr`). Is it weak, moderate, or strong?
3. Because income data is typically right-skewed, create a new column `log_income =
   np.log(labor_income)`. Plot histograms of `labor_income` and `log_income`
   side-by-side and compare their skewness (`.skew()`).
4. Re-make the scatterplot and correlation using `log_income` instead. Did the
   relationship become more linear?

In [ ]:
# TODO: scatterplot of education_years vs labor_income


In [ ]:
# TODO: Pearson correlation of education_years and labor_income


In [ ]:
# TODO: create log_income, compare skew and histograms


In [ ]:
# TODO: scatterplot + correlation of education_years vs log_income


**Which outcome variable will you model, and why?** *(replace this text)*

## Step 4 — Outlier Treatment

**Instructions:**
1. Make box plots of `log_income`, `education_years`, and `age` in `analysis_df`.
2. Write a helper function `iqr_bounds(series, k=1.5)` that returns the lower/upper IQR
   fences for a series.
3. Count how many rows fall outside the IQR fences for `log_income`.
4. Create `clean_df` by removing those outlier rows. Print sample sizes before and
   after.

In [ ]:
# TODO: box plots for log_income, education_years, age


In [ ]:
# TODO: define iqr_bounds() and use it to find/remove outliers -> clean_df


## Step 5 — Train/Test Split

**Instructions:**
Split `clean_df` into `train` (80%) and `test` (20%) using `train_test_split` with a
fixed `random_state` for reproducibility. Print the size of each set.

In [ ]:
# TODO: train, test = train_test_split(...)


## Step 6 — Building a Simple Linear Regression Model

**Instructions:**
Using `statsmodels.formula.api.ols`, fit a simple linear regression of the form:

$$\text{log\_income} = \beta_0 + \beta_1 \times \text{education\_years} + \varepsilon$$

on the `train` set. Save the fitted model as `model_simple` and print its `.summary()`.

In [ ]:
# TODO: model_simple = smf.ols(...).fit()
# TODO: print(model_simple.summary())


## Step 7 — Quantifying Model Fit

**Instructions:**
1. Extract the R-squared value from `model_simple`.
2. Compute the Residual Standard Error (RSE) — the square root of `model_simple.mse_resid`.
3. In a sentence, explain what each number tells you about the model.

In [ ]:
# TODO: compute and print R-squared and RSE


## Step 8 — Checking Model Residuals

**Instructions:**
1. Get the fitted values and residuals from `model_simple`.
2. Make a "residuals vs. fitted" scatterplot with a horizontal reference line at 0.
3. Make a Normal Q-Q plot of the residuals (`statsmodels.api.qqplot`).
4. Based on both plots, do the residuals look reasonably well-behaved (roughly constant
   spread, roughly normal)? Explain.

In [ ]:
# TODO: residuals vs fitted plot


In [ ]:
# TODO: Q-Q plot


**Your interpretation:** *(replace this text)*

## Step 9 — Visualizing Model Fit

**Instructions:**
Make a single plot showing (a) the scatter of `education_years` vs `log_income` in the
training data, (b) the fitted straight-line regression, and (c) a non-parametric LOWESS
smoother for comparison (`seaborn.regplot(..., lowess=True)`). Do the two lines agree?

In [ ]:
# TODO: combined scatter + linear fit + LOWESS plot


## Step 10 — Reading and Interpreting the Model Results

**Instructions:**
1. Extract $\beta_0$ (intercept) and $\beta_1$ (slope on `education_years`) from
   `model_simple`.
2. Extract the p-value and 95% confidence interval for $\beta_1$.
3. Because the outcome is `log(income)`, convert $\beta_1$ into an approximate
   **percentage** effect using $(e^{\beta_1} - 1) \times 100$. Write one sentence
   interpreting this number in plain English.

In [ ]:
# TODO: extract beta0, beta1, p-value, confidence interval


In [ ]:
# TODO: convert beta1 to a percentage effect and print an interpretation


## Step 11 — Making Predictions and Evaluating on the Test Set

**Instructions:**
1. Use `model_simple.predict()` to generate `pred_log_income` for the `test` set, and
   exponentiate it to get `pred_income`.
2. Compute the test-set R-squared and RMSE (on the log scale).
3. Compute the RMSE of a **naive baseline** that always predicts the training mean of
   `log_income`. Does your model beat the baseline? By how much?

In [ ]:
# TODO: generate predictions on the test set


In [ ]:
# TODO: compute test R-squared, RMSE, and compare to a naive baseline


## Step 12 — Multiple Linear Regression

**Instructions:**
Extend the model to include additional predictors:

$$\text{log\_income} = \beta_0 + \beta_1\,\text{education\_years} + \beta_2\,\text{age} + \beta_3\,\text{age}^2 + \beta_4\,\text{gender} + \beta_5\,\text{married} + \beta_6\,\text{degree\_tier} + \varepsilon$$

1. Create an `age_sq` column (age squared) in both `train` and `test`.
2. Fit this multiple regression using `statsmodels.formula.api.ols` with a formula
   string. Use `C(...)` around categorical columns so statsmodels dummy-encodes them
   automatically. Save the model as `model_multi` and print its summary.
3. Why might we want to include `age` **and** `age²` together, instead of just `age`?

In [ ]:
# TODO: create age_sq for train and test


In [ ]:
# TODO: fit model_multi with the extended formula, print summary


## Step 13 — Assessing the Multiple Regression Model

**Instructions:**
1. Compare the R-squared **and adjusted R-squared** of `model_simple` vs `model_multi`.
2. Run a nested-model F-test (`statsmodels.stats.anova.anova_lm`) comparing the two
   models. Is the improvement statistically significant?
3. Check for multicollinearity using **Variance Inflation Factors (VIF)** on the
   multiple model's predictors (`statsmodels.stats.outliers_influence
   .variance_inflation_factor`). Are any VIFs concerningly high (rule of thumb: > 5–10)?

In [ ]:
# TODO: compare R-squared / adjusted R-squared for both models


In [ ]:
# TODO: nested F-test with anova_lm


In [ ]:
# TODO: compute VIFs for the multiple model's predictors


## Step 14 — Interpreting the Multiple Regression Coefficients

**Instructions:**
1. Build a small table of `model_multi`'s coefficients, their percentage-effect
   equivalents (like Step 10), and their p-values.
2. Evaluate `model_multi` on the test set (R-squared and RMSE) and compare to
   `model_simple`'s test performance.
3. Write 3–4 sentences interpreting the practical meaning of the key coefficients
   (education, age/age², gender, marital status, degree tier). Remember that
   categorical coefficients are relative to a left-out reference category.

In [ ]:
# TODO: coefficient table with percentage effects and p-values


In [ ]:
# TODO: evaluate model_multi on the test set


**Your interpretation:** *(replace this text)*

## Step 15 — Limitations and Next Steps

**Instructions:** In your own words, write a short "limitations" section addressing at
least these questions:
1. Is `analysis_df` a random sample of all workers, or a specific sub-population? What
   does that mean for how far you can generalize your conclusions?
2. Does a significant coefficient on `education_years` prove education *causes* higher
   income? Why or why not?
3. Name one thing about this dataset's coding (e.g., `educated_in_us`) that limited what
   you could confidently conclude.
4. Suggest one concrete next step (a new variable, a different model type, more data)
   that would strengthen this analysis.

*(write your limitations discussion here)*

## Step 16 — Executive Summary

**Instructions:** Write a 4–6 bullet executive summary of your project, written for a
non-technical audience: what question you asked, what you found, how confident you are,
and what you'd want to do next.

*(write your executive summary here)*